# 4-7 速度比較: VBA vs Python

本講座のラストです。**同じ集計を VBA と Python で書いて、所要時間を比較** します。

題材:

> **47 都道府県分の `sales_2026-01_XX.xlsx` (各 1,000 行) を全部読み込み、商品別の売上合計を出す**

Python なら 4-2 でやった処理そのまま。VBA で書くとどうなるか、そして速度はどれくらい違うか。**ファイルを開かない pandas の威力** をここで体感してください。

## なぜ速度差が出るのか

### VBA の場合

```
for ファイル in 47ファイル:
    Excel アプリでファイルを開く        ← (重い: 描画/計算エンジン起動)
    全行をスキャンして集計
    Excel アプリでファイルを閉じる
```

- **`Workbooks.Open` が 1 回あたり数百ミリ秒〜数秒**
- これを 47 回繰り返す
- Excel プロセスが裏で動いているので、CPU と メモリも食う

### pandas (Python) の場合

```
for ファイル in 47ファイル:
    pandas が xlsx (= ZIP) を直接読み込む   ← (軽い: Excel アプリ不要)
    DataFrame に格納
pd.concat で束ねて groupby で集計
```

- **`pd.read_excel` は ZIP/XML を直接パース** するだけ
- **Excel アプリは起動しない**
- ファイル開閉の重さがゼロに近い

👉 これが、ファイル数が増えれば増えるほど **差が開いていく** 理由です。

## 計測 1: Python 側

`%%time` という Jupyter の機能を使うと、**セルの実行時間を自動計測** してくれます。

**注意**: 公平な比較のため、VBA と同じく **「47 ファイル読込 + 商品別集計」だけ** を計測します (マスタ結合やグラフ作成は含まない)。

In [ ]:
# === Colab ===
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/python-data-basics"

# === ローカル ===
BASE = "."

import pandas as pd
from pathlib import Path

SALES_DIR = f"{BASE}/data/capstone/sales_2026-01"

In [ ]:
%%time
# Python: 47 ファイル読み込み + 商品別集計
files = sorted(Path(SALES_DIR).glob("sales_*.xlsx"))
dfs = [pd.read_excel(f, sheet_name="売上") for f in files]
all_sales = pd.concat(dfs, ignore_index=True)
result_python = all_sales.groupby("product_name")["amount"].sum().sort_values(ascending=False)
print(result_python)

**`Wall time:` の値**（最後の行の数字）が **Python の所要時間** です。

目安: ローカルなら **2〜10 秒**、Colab でも **5〜20 秒** 程度。

## 計測 2: VBA 側 — 手順

VBA は Excel の中でしか動かないので、**Colab では実行不可** です。Windows or Mac の **Excel アプリ** で実行してください。

用意してあるのは:

```
04_capstone/vba/aggregate_monthly.bas
```

**Python と同じことを VBA で書いたマクロ** です。中身は `Workbooks.Open` で 47 ファイルを順に開き、行ループで集計するシンプルな実装。`Application.ScreenUpdating = False` 等の **高速化のおまじないも入れた「最適化済み」版** にしてあります。

### 取り込み手順 (Windows / Mac 共通)

1. 任意の場所に **新しい Excel ブック** を作る → 名前は何でも OK
2. 「名前を付けて保存」で **拡張子を `.xlsm`** (マクロ有効ブック) に変えて保存
3. **`Alt + F11`** (Mac は **`Option + F11`** または `Fn + Option + F11`) で **VBE (Visual Basic Editor)** を開く
4. VBE の上部メニュー: **`ファイル` → `ファイルのインポート`** から `aggregate_monthly.bas` を選択
5. 左ペインの `モジュール` の下に **`Module_Aggregate`** が増えたことを確認
6. その中の **`AggregateMonthlySales`** にカーソルを置いて、**`F5`** で実行
7. ダイアログに **`sales_2026-01` フォルダの絶対パス** を入れる
   - Windows 例: `C:\Users\YourName\Desktop\python-data-basics-local\data\capstone\sales_2026-01`
   - Mac 例    : `/Users/yourname/python-data-basics-local/data/capstone/sales_2026-01`
8. 集計が走り、完了時に **MsgBox で経過時間が表示** される

> 💡 **「開発」タブの表示**: Excel の「ファイル → オプション → リボンのユーザー設定 → 開発」にチェック。これがあると VBE を画面ボタンからも開けます。

### 計測中、画面で何が起きるか

`ScreenUpdating = False` を入れているので画面はチカチカしませんが、**裏では Excel が 47 ファイルを次々開いて閉じている** はずです。

- タスクマネージャ (Windows) や アクティビティモニタ (Mac) を眺めると、**Excel のメモリ使用量が一気に増える** のが見えます
- 1 ファイルあたり「開く → 読む → 閉じる」を 47 回繰り返している

目安: ローカル PC のスペックにもよりますが、**30 秒〜2 分** 程度。

## 結果を記録する

下のセルの数値を、自分の環境で計測した値に書き換えて実行してください。

In [ ]:
# あなたの計測結果
python_sec = 5.0     # ← 上で測った Wall time に書き換え
vba_sec    = 60.0    # ← MsgBox に出た秒数に書き換え

ratio = vba_sec / python_sec
print(f"Python : {python_sec:>6.2f} 秒")
print(f"VBA    : {vba_sec:>6.2f} 秒")
print(f"差     : Python は VBA の約 {ratio:.0f} 倍速い")

## 解説 — なぜこの差なのか

VBA がやっていること:

1. **Excel アプリで 1 ファイルずつ開く** ← ここが大半の時間
2. 全シート・全行を Excel のメモリ上にロード
3. 計算エンジンが起動して数式を再計算 (たとえ数式が無くても)
4. VBA でセルを 1 つずつ読む
5. 閉じる

Python (pandas) がやっていること:

1. **xlsx は中身が ZIP + XML**
2. pandas (内部の openpyxl) が **ZIP を直接展開して XML を読む**
3. NumPy の配列に直接ロード
4. Excel アプリは起動していない

つまり、**VBA は「Excel アプリの上で動いている」** のに対して、**Python は「ファイルを直接読む別経路」を持っている**。

## 「最適化 VBA」と書いていますが…

今回の `aggregate_monthly.bas` には、すでに **VBA の代表的な高速化テクニック** が入っています:

- `Application.ScreenUpdating = False` (画面更新を止める)
- `Application.Calculation = xlCalculationManual` (再計算を止める)
- `Application.DisplayAlerts = False` (警告ダイアログを抑制)
- セルを `Range.Value` で **配列に一括取得** してからループ (1 セルずつアクセスより圧倒的に速い)

**これだけ最適化しても、Python の数倍〜数十倍遅い**。

つまり、**「VBA の書き方が悪い」のではなく「Excel アプリ越しにファイルを触る」という仕組みが遅い**、ということです。

## ファイル数が増えたら？

今回は 47 ファイル × 1,000 行 = 47,000 行でしたが、もし **支店が増えて 1,000 ファイル**、**1 年分 12,000 ファイル**、**全国チェーンで毎日上がる 30,000 ファイル** になったら?

- **VBA**: ファイル数に比例してほぼ線形に時間が増える。**夜間バッチで 1 時間** とかザラ
- **Python**: ファイル開閉が軽いので、ファイル数が増えても時間の増え方が緩やか。**数十秒〜数分**

**「ファイルを開かない」がいかに強いか**。これが Python を業務に取り入れる一番の動機です。

## 4-6 の関数全体だとどれくらい?

せっかくなので、4-6 で作った **「ファイル読込 → 結合 → 集計 → グラフ → Excel 書出」までの全部** も計測してみましょう。

**VBA で同じことをやろうとすると、Excel ファイル書き出しやグラフ描画も含まれる** ので、もはや時間の桁が違ってきます (試しに書こうと思うと、たぶん丸 1 日かかる)。

In [ ]:
%%time
# 4-6 の関数を再定義 (このノート単体で動かすため)
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import japanize_matplotlib
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

# 関数定義は 4-6 のものをそのまま — 紙幅の都合でここでは省略し、
# 4-6 ノートを先に実行しておくか、関数定義部分をコピペしてから実行してください。
# その上で:
# generate_monthly_report("2026-01", base=BASE)

# 簡易: 4-2〜4-3 相当の処理だけ計測
files = sorted(Path(SALES_DIR).glob("sales_*.xlsx"))
dfs = []
for f in files:
    d = pd.read_excel(f, sheet_name="売上")
    d["prefecture_code"] = int(f.stem.split("_")[2])
    dfs.append(d)
all_sales = pd.concat(dfs, ignore_index=True)

mp = pd.read_excel(f"{BASE}/data/capstone/master_products.xlsx")
mb = pd.read_excel(f"{BASE}/data/capstone/master_branches.xlsx")
df = all_sales.merge(mp[["product_code","category","cost"]], on="product_code").merge(mb, on="prefecture_code")
df["profit"] = df["amount"] - df["cost"] * df["quantity"]

by_region = df.groupby("region")["amount"].sum()
by_pref   = df.groupby("prefecture_name")["amount"].sum()
by_prod   = df.groupby("product_name")["amount"].sum()
print("集計完了")

## まとめ

- VBA は **Excel アプリでファイルを開いて触る** 仕組み → ファイル開閉が重い
- Python (pandas) は **xlsx ファイルを直接パース** → Excel アプリを起動しない
- 最適化 VBA でも、**Python は数倍〜数十倍速い**
- ファイル数が増えるほど **差は線形に開いていく**
- ファイルを開かない = メモリ・CPU の使用量も小さい = **大量データに耐える**

事務職の業務で「VBA で何時間もかかっていた集計が、Python で数秒で終わる」のは、書き方の問題ではなく **仕組みの問題** です。Python を覚えることで、**「待ち時間」がそのまま「他の仕事に使える時間」** に変わります。

## 講座を終えて

本講座お疲れさまでした。ここまでで:

- **第 1 章**: Python の文法 (変数・型・リスト・辞書・分岐・ループ・関数)
- **第 2 章**: pandas で Excel/CSV を扱う (読込・集計・絞り込み・加工・書出)
- **第 3 章**: matplotlib で可視化 (棒・折れ線・散布図・体裁の整え方)
- **第 4 章**: 47 支店の月次レポートを自動生成する実務スクリプト + VBA 比較

を一通り学びました。**毎月の手作業を Python に置き換える** ことができる状態です。

次の一歩として:

- **自分の業務データ** で 4-6 の関数を書き換えてみる (列名 / シート名 / フォルダ名を自分の業務に合わせる)
- **エラー処理** (try/except) を追加して、ファイルが無いときに分かりやすいメッセージを出す
- **メール送信** や **Teams 通知** と組み合わせて、レポートを自動で関係者に配る
- **複数月の比較** (2026-01 と 2026-02 を並べる) を関数化する

ご質問・改善提案は **ストアカのメッセージ機能** からお寄せください。Happy automating 🎉